---
numbering: false
---

# 1.2. Loss Functions and the Constant Model

## Motivation
Suppose that you're looking for an off campus apartment for next year. Unfortunately, none of them are in your price range, so you decide to live with your parents in Detroit and commute. To see if you can save some time on the road each day, you keep track of how long it takes for you to get to school. Here's a preview of the commute times dataset. 

<img src="02-loss-functions-constant-model-imgs/preview.png">

We'll mainly focus on the ``departure_hour`` and ``minutes`` columns, which refer to time leaving the house and total time commuting respectively. Since our goal is to save time, we want to predict the value of ``minutes``, so it will be our $y$ variable. We'll use ``departure_hour`` to predict our commute time, so it will be our $x$ variable. Let's see if we can gain any insights from looking at the distribution of commute times in our dataset:

<img src="02-loss-functions-constant-model-imgs/hist-minutes.png">

It seems like a typical commute is somewhere between 60 and 80 minutes, but there are outliers. Some of the commutes have taken closer to two hours, and some have been under an hour. We're using ``departure_hour`` to predict ``minutes``, so let's visualize the relationship between them with a scatterplot (recall that a scatterplot is used for quantitative data):

<img src="02-loss-functions-constant-model-imgs/scatter-1.png">

We can observe a general downward trend: the later the departure time, the lower the commute time. Now, let's properly establish the problem to make sure we're all on the same page.

__GOAL__: Predict commute time

Problem Type: Regression - we want to predict ``minutes``, which is quantitative (or numerical) data

How to Solve: learn a pattern from the data

However, there's an important assumption we have to make for our predictions to be useful. Let's think about _why_ we're trying to predict commute time in the first place. We don't need predict the time for previous commutes, since we already know how long they took. We want to predict the time needed for _future_ commutes; ones that we don't know about yet. Therefore, we need to _assume that the future resembles our data from the past_. Another way of saying this is that we want the model to generalize well to unseen data. For example, if a new highway between Detroit and Ann Arbor gets built, the patterns previously learned will no longer be generalizable because the unseen data no longer resembles the past.  Keep the concept of generalizability in mind for later in the class, it will be much more important then.

## Models
You've probably heard the word "model" get thrown around a lot in the past few years, so let's give it a more concrete definition.

:::{note} Definition: Model
A __model__ is a set of assumptions about how data were generated.
:::

A common saying in machine learning that teaches us some important ideas about models is "all models are wrong, but some are useful":
1. Models cannot be 100% correct no matter how accurate we make them.
2. A model is useful, even if it won't always be correct. Example use cases are approximating or calculating.

A more subtle idea hidden in this statement relates to the relationship between complexity and correctness. Models can be very complex taking in several inputs and performing various mathematical operations to produce a prediction. In fact, later in this class you will learn about and design some rather sophisticated models yourself! However, there's diminishing returns because the model can never be 100% correct. You can usually get decent predictions using simple models, which can still come in handy.

Returning to the scatterplot above, it seems reasonable for our model to be a line of best fit. However, it would be impossible to draw a line that goes through every single point, so we have to decide which data points are more valuable for making a prediction.

## Hypothesis Functions

:::{note} Definition: Hypothesis Function and Notation
- $x$: "input", "independent variable", or "feature"
- $y$ "response", "dependent variable", or "target"

The $i$th observation/data point is denoted $(x_i, y_i)$
Note that each $x_i$ is used to predict it's respective $y_i$

A hypothesis function $H$ takes in an $x_i$ as an input and returns a predicted $y_i$.
:::

Each hypothesis function has parameters, which define the relationship between the input and output. For example, the constant model $H(x)=h$ has only one parameter, $h$.

$H(x)=60$             |$H(x)=70$     | $H(x)=100$     
:-------------------------:|:-------------------------:|:-------------------------:|
<img src="02-loss-functions-constant-model-imgs/constant-60.png"> | <img src="02-loss-functions-constant-model-imgs/constant-70.png"> | <img src="02-loss-functions-constant-model-imgs/constant-100.png">

The graph of the constant model is a flat line, since it will always predict $h$ for any $x_i$. Visually, it's rather obvoius that $h=100$ would be a poor choice since it's so far from most of the data. $h=60$ and $h=70$ seem like much more reasonable predictions, but how we can quantify which one is better? Let's try with a small example first.

Suppose we have a small dataset of only 5 commute times.
$$y_1=72 \newline y_2=90 \newline y_3=61 \newline y_4=85 \newline y_5=92$$
Summary statistics such as the mean or median both seem like reasonable choices, but which one is better? It's difficult to tell without some metric to compare them with, so let's learn about one.

:::{note} Definition: Loss functions
A loss function quantifies how bad the prediction is for a single data point.
- If our prediction is __close__ to the actual value, we should have __low__ loss
- If our prediction is __far__ from the actual value, we should have __high__ loss
:::
A common starting point for a loss function is error, defined as the difference between actual and predicted values.
$$e_i={\color{blue}y_i}-{\color{orange}H(x_i)}$$
where ${\color{blue}y_i}$ is the <span style="color:blue">actual value</span> and ${\color{orange}H(x_i)}$ is the <span style="color:orange">predicted value</span>. Suppose the actual commute time $y_i=80$.
- If I predict 75, $e_i=80-75=5$
- If I predict 72, $e_i=80-72=8$
- If I predict 100, $e_i=80-100=-20$

A lower error is better, so 75 is a better prediction than 72. But what about 100, which has a negative error? It's difficult to interpret and compare loss with different signs, so we want to choose a loss function which can't have negative outputs, such as by squaring or taking the absolute value.

### Squared Loss
One such loss function is squared loss $L_{sq}$, which computes $({\color{blue}\text{actual}}-{\color{orange}\text{predicted}})^2$
$$L_{sq}({\color{blue}y_i}, {\color{orange}H(x_i)})=({\color{blue}y_i}-{\color{orange}H(x_i)})^2$$

You may be thinking that the intuitive option for removing negative values is absolute loss $L_{abs}(y_i, H(x_1))=|x|$, so why did we choose squared loss?
- The resulting function is differentiable (more on this later)
- Theoretical relationship to the normal distribution in statistics

Choosing a loss function is an important step when designing a model, and one we'll go more in depth on later this semester. For now, let's try applying squared loss to each data point in the small example, with the median $h=85$ as or prediction.

$$
\begin{align*}
L_{sq}(y_1)=(72-85)^2=(-13)^2=169 \newline L_{sq}(y_2)=(90-85)^2=5^2=25 \newline L_{sq}(y_3)=(61-85)^2=(-24)^2=576\newline L_{sq}(y_4)=(85-85)^2=0 \newline L_{sq}(y_5)=(92-85)^2=7^2=49
\end{align*}$$

Each output of $L_{sq}$ tells us how good our prediction was at that specific data point, but we'd like a single number which describes the quality of our predictions across the whole dataset. That way we can compare the numbers to determine which model is better. One way to compute this is taking the average of the squared losses.

- For the median, $h = {\color{purple}{85}}$:

  $$\frac{1}{5} \left( (72 - {\color{purple}{85}})^2 + (90 - {\color{purple}{85}})^2 + (61 - {\color{purple}{85}})^2 + (85 - {\color{purple}{85}})^2 + (92 - {\color{purple}{85}})^2 \right) = \boxed{163.8}$$

- For the mean, $h = {\color{purple}{80}}$:

  $$\frac{1}{5} \left( (72 - {\color{purple}{80}})^2 + (90 - {\color{purple}{80}})^2 + (61 - {\color{purple}{80}})^2 + (85 - {\color{purple}{80}})^2 + (92 - {\color{purple}{80}})^2 \right) = \boxed{138.8}$$

The mean is a better prediction because it has a lower average squared loss! Another term for average squared loss is mean squared error (MSE), and it will be another important concept throught this course.

## Mean Squared Error
Let's start by generalizing mean squared error to any prediction $h$ for our small commute times dataset.
$$R_\text{sq}(h) = \frac{1}{5} \left( (72 - h)^2 + (90 - h)^2 + (61 - h)^2 + (85 - h)^2 + (92 - h)^2 \right)$$

Our result is a function that takes in any prediction $h$ and outputs the mean squared error, which is a measurement of the quality of the prediction. We can pick any $h$ as our prediction, but we want the output of $R_\text{sq}(h)$ to be as small as possible. The function is notated as $R_{sq}$, where the $R$ stands for risk, as in "empirical risk". Don't worry about this term for the time being, just keep in mind that $R$ is always the function for average loss.

:::{tip} Activity x
:class: dropdown
Plotting the function $R_\text{sq}(h)$ as defined above gives us this graph.
<img src="02-loss-functions-constant-model-imgs/risk.png">
Explain the significance of the value of $h$ at the vertex
:::

We can generalize mean squared error more, for any prediction $h$ and any set of actual values $y_1,y_2,...,y_n$
$$R_\text{sq}(h)=\frac{1}{n} \left((y_1- h)^2 + (y_2 - h)^2 ... + (y_n - h)^2 \right)
 = \frac{1}{n}\sum_{i=1}^n(y_i-h)^2$$

It might be confusing to see so many symbols, but here $h$ is our only variable because we know the actual values in the dataset. 

Our goal is now to find the $h$ which _minimizes_ $R_\text{sq}(h)$, called this value $h^*$. Time for some differentiating. 

TODO: activity or walkthrough of derivation, leaning towards act since it should be review